In [60]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/varunrajput/beed-bangalore-eeg-epilepsy-dataset/BEED_Data.csv


In [61]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [62]:
df = pd.read_csv("/kaggle/input/datasets/varunrajput/beed-bangalore-eeg-epilepsy-dataset/BEED_Data.csv")


In [63]:
X = df.drop("y", axis=1)
y = df["y"]


In [64]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [65]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [66]:
# LASSO
lasso_model = LogisticRegression(
    penalty="l1",
    solver="saga",
    C=1.0,
    max_iter=5000
)

lasso_model.fit(X_train, y_train)

# Predictions
l1_train_pred = lasso_model.predict(X_train)
l1_test_pred = lasso_model.predict(X_test)

# ================= SPARSITY CHECK =================

coef = lasso_model.coef_[0]

non_zero = np.sum(coef != 0)
total = len(coef)

print("\n===== SPARSITY ANALYSIS =====")
print("Non-zero features:", non_zero)
print("Total features:", total)
print("Sparsity (%):", (non_zero / total) * 100)



===== SPARSITY ANALYSIS =====
Non-zero features: 15
Total features: 16
Sparsity (%): 93.75


In [67]:
#RIDGE
ridge_model = LogisticRegression(
    penalty="l2",
    solver="lbfgs",
    C=1.0,
    max_iter=5000
)

ridge_model.fit(X_train, y_train)

l2_train_pred = ridge_model.predict(X_train)
l2_test_pred = ridge_model.predict(X_test)



In [68]:
#elastic net
elastic_model = LogisticRegression(
    penalty="elasticnet",
    solver="saga",
    l1_ratio=0.5,
    C=1.0,
    max_iter=5000
)

elastic_model.fit(X_train, y_train)

en_train_pred = elastic_model.predict(X_train)
en_test_pred = elastic_model.predict(X_test)

coef = elastic_model.coef_[0]


print("===== L1 (LASSO) =====")
print("Train Accuracy:", accuracy_score(y_train, l1_train_pred))
print("Test Accuracy:", accuracy_score(y_test, l1_test_pred))

print("\n===== L2 (RIDGE) =====")
print("Train Accuracy:", accuracy_score(y_train, l2_train_pred))
print("Test Accuracy:", accuracy_score(y_test, l2_test_pred))

print("\n===== ELASTIC NET =====")
print("Train Accuracy:", accuracy_score(y_train, en_train_pred))
print("Test Accuracy:", accuracy_score(y_test, en_test_pred))

===== L1 (LASSO) =====
Train Accuracy: 0.485625
Test Accuracy: 0.4575

===== L2 (RIDGE) =====
Train Accuracy: 0.481875
Test Accuracy: 0.454375

===== ELASTIC NET =====
Train Accuracy: 0.4821875
Test Accuracy: 0.455625
